In [1]:
import torch
from models.magnifier import MagnifierModel as magnifier

In [4]:
# ------------------------------------------------------------------
# Usage Examples and Testing
# ------------------------------------------------------------------

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Testing Enhanced 2D FNO Magnifier on device: {device}\n")
    print("=" * 70)

    # ------------------------------------------------------------------
    # Example 1: Enhanced Configuration with Progressive Expansion
    # ------------------------------------------------------------------
    print("\n[Example 1] Enhanced Model with Progressive Channel Expansion")
    print("-" * 70)

    # Typical real-world shapes
    nb = 32               # batch size
    N = 5                # coarse patch size
    f = 4                # upscaling factor
    P_fine = N * f       # 40
    nt = 88              # time steps
    C_in = 2             # channels: coarse u interp + fine bed

    # Create dummy patch input
    patch_input = torch.randn(nb, C_in, P_fine, P_fine, nt, device=device)



    model_enhanced = magnifier(
        in_channels=C_in,
        base_channels=32,
        num_fno_blocks=4,
        fno_modes_x=6,
        fno_modes_y=6,
        num_refinement_blocks=4,
        num_residual_per_block=3,
        channel_multipliers=[1.0, 1.5, 2, 2],  # 48→64→80→96→96
        dropout=0.1,
        use_attention=False,
        use_pyramid_pooling=True,
        use_gradient_checkpointing=False
    ).to(device)
    
    output = model_enhanced(patch_input)
    
    print(f"Input shape:  {patch_input.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Channel progression: {model_enhanced.channel_counts}")
    
    total_params = sum(p.numel() for p in model_enhanced.parameters())
    print(f"Total parameters: {total_params:,}")
    torch.cuda.empty_cache()


Testing Enhanced 2D FNO Magnifier on device: cuda


[Example 1] Enhanced Model with Progressive Channel Expansion
----------------------------------------------------------------------
Input shape:  torch.Size([32, 2, 20, 20, 88])
Output shape: torch.Size([32, 1, 20, 20, 88])
Channel progression: [32, 32, 48, 64, 64]
Total parameters: 2,441,087


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Simple3DMagnifier(nn.Module):
    """
    A simple 3D Convolutional operator for spatial refinement.
    Input:  [nb, 2, P_fine, P_fine, nt]
    Output: [nb, 1, P_fine, P_fine, nt]
    """
    def __init__(self, in_channels=2, width=32):
        super().__init__()
        
        # 1. Feature Extraction (3D Convolutions)
        # kernel_size=3 with padding=1 keeps dimensions the same
        self.conv1 = nn.Conv3d(in_channels, width, kernel_size=3, padding=1)
        self.conv2 = nn.Conv3d(width, width, kernel_size=3, padding=1)
        self.conv3 = nn.Conv3d(width, width, kernel_size=3, padding=1)
        
        # 2. Nonlinearity
        self.act = nn.GELU()
        
        # 3. Final Projection to 1 channel
        self.final = nn.Conv3d(width, 1, kernel_size=1)

    def forward(self, x):
        # x: [nb, 2, h, w, t]
        
        # Layer 1
        x = self.act(self.conv1(x))
        
        # Layer 2 (Residual Connection)
        residual = x
        x = self.act(self.conv2(x))
        x = x + residual # Helps with gradient flow
        
        # Layer 3
        x = self.act(self.conv3(x))
        
        # Output Projection
        x = self.final(x)
        
        return x

In [3]:
# ------------------------------------------------------------------
# Usage Examples and Testing for Simple3DMagnifier
# ------------------------------------------------------------------

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Testing Simple 3D Magnifier on device: {device}\n")
    print("=" * 70)

    # ------------------------------------------------------------------
    # Configuration
    # ------------------------------------------------------------------
    # Typical shapes based on your project details
    nb = 32               # Batch size
    P_fine = 40           # Fine patch size (N*f)
    nt = 88               # Time steps
    C_in = 2              # Channels: Coarse u interp + Fine bed

    # Create dummy patch input: [nb, C_in, H, W, T]
    patch_input = torch.randn(nb, C_in, P_fine, P_fine, nt, device=device)

    # Initialize the simplified 3D Magnifier
    # We use width=32 to stay within V100/A40 memory limits while providing 
    # enough capacity for time-series forecasting
    model_simple = Simple3DMagnifier(
        in_channels=C_in,
        width=32
    ).to(device)
    
    # Forward pass
    output = model_simple(patch_input)
    
    print(f"Input shape:  {patch_input.shape}")  # Expected: [32, 2, 40, 40, 88]
    print(f"Output shape: {output.shape}")     # Expected: [32, 1, 40, 40, 88]
    
    total_params = sum(p.numel() for p in model_simple.parameters())
    print(f"Total parameters: {total_params:,}")
    
    # Cleanup for GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Testing Simple 3D Magnifier on device: cuda

Input shape:  torch.Size([32, 2, 40, 40, 88])
Output shape: torch.Size([32, 1, 40, 40, 88])
Total parameters: 57,153


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualDenseBlock3d(nn.Module):
    """
    A deeper block that allows features to bypass layers, 
    preserving the 'interpolated' coarse physics while adding 
    high-resolution details from the bed topography.
    """
    def __init__(self, nf=32, gc=16):
        super().__init__()
        # gc = growth channel
        self.conv1 = nn.Conv3d(nf, gc, 3, 1, 1)
        self.conv2 = nn.Conv3d(nf + gc, gc, 3, 1, 1)
        self.conv3 = nn.Conv3d(nf + 2 * gc, nf, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.conv3(torch.cat((x, x1, x2), 1))
        return x3 * 0.2 + x  # Residual scaling for stability

class Deep3DMagnifier(nn.Module):
    def __init__(self, in_channels=2, width=48, num_blocks=4):
        super().__init__()
        
        # 1. Initial Feature Extraction
        self.lifting = nn.Conv3d(in_channels, width, kernel_size=3, padding=1)
        
        # 2. Deep Residual Body
        # Each block refines the spatial/temporal features
        self.res_blocks = nn.ModuleList([
            ResidualDenseBlock3d(nf=width, gc=width//2) 
            for _ in range(num_blocks)
        ])
        
        # 3. Global Skip Connection Fusion
        self.trunk_conv = nn.Conv3d(width, width, kernel_size=3, padding=1)
        
        # 4. Final Projection to 1 channel (Water Depth/Velocity)
        self.projection = nn.Sequential(
            nn.Conv3d(width, width, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv3d(width, 1, kernel_size=1)
        )

    def forward(self, x):
        # x: [nb, 2, P_fine, P_fine, nt]
        feat = self.lifting(x)
        
        # Run through the deep residual body
        res = feat
        for block in self.res_blocks:
            res = block(res)
        
        # Global skip connection: helps the model keep the base physics 
        # from the interpolation while adding the learned details
        res = self.trunk_conv(res)
        out = feat + res
        
        return self.projection(out)

In [8]:
# ------------------------------------------------------------------
# Usage Examples and Testing for Deep 3DMagnifier
# ------------------------------------------------------------------

if __name__ == "__main__":
    # Check for GPU (Recommended for your V100/A40 setup)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Testing Deep 3D Magnifier on device: {device}\n")
    print("=" * 70)

    # ------------------------------------------------------------------
    # Configuration
    # ------------------------------------------------------------------
    # Matching your specific project requirements
    nb = 32               # Batch size
    P_fine = 40           # Upscaled spatial resolution (N * f)
    nt = 88               # Total time steps for Hurricane Matthew
    C_in = 2              # Channel 1: Coarse Interp, Channel 2: Fine Bed

    # Create dummy patch input: [Batch, Channels, Height, Width, Time]
    # This represents your high-resolution grid for learned interpolation
    patch_input = torch.randn(nb, C_in, P_fine, P_fine, nt, device=device)

    # Initialize the Deep 3D Magnifier
    # width=48 and num_blocks=4 provide the 'depth' to capture complex physics
    model_deep = Deep3DMagnifier(
        in_channels=C_in,
        width=48,
        num_blocks=4
    ).to(device)
    
    # Forward pass
    # This simulates the learnable interpolation/refinement process
    output = model_deep(patch_input)
    
    # ------------------------------------------------------------------
    # Results Summary
    # ------------------------------------------------------------------
    print(f"Input shape:  {patch_input.shape}")  # [32, 2, 40, 40, 88]
    print(f"Output shape: {output.shape}")     # [32, 1, 40, 40, 88]
    
    total_params = sum(p.numel() for p in model_deep.parameters())
    print(f"Total parameters: {total_params:,}")
    
    # Performance check: Output should match the target spatial/temporal grid
    assert output.shape == (nb, 1, P_fine, P_fine, nt), "Output shape mismatch!"
    
    # Cleanup for GPU memory to keep your mgc-mri nodes healthy
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print("\nTest passed: Model is deep enough for refinement and fits in memory.")

Testing Deep 3D Magnifier on device: cuda

Input shape:  torch.Size([32, 2, 40, 40, 88])
Output shape: torch.Size([32, 1, 40, 40, 88])
Total parameters: 936,289

Test passed: Model is deep enough for refinement and fits in memory.


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualDenseBlock3d(nn.Module):
    """
    A deeper block that allows features to bypass layers, 
    preserving the 'interpolated' coarse physics while adding 
    high-resolution details from the bed topography.
    """
    def __init__(self, nf=32, gc=16):
        super().__init__()
        # gc = growth channel
        self.conv1 = nn.Conv3d(nf, gc, 3, 1, 1)
        self.conv2 = nn.Conv3d(nf + gc, gc, 3, 1, 1)
        self.conv3 = nn.Conv3d(nf + 2 * gc, nf, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.conv3(torch.cat((x, x1, x2), 1))
        return x3 * 0.2 + x  # Residual scaling for stability


class MultiStreamMagnifier(nn.Module):
    def __init__(self, width=48, num_blocks=4):
        super().__init__()
        
        # Branch A: Water Depth (Lower Magnitude)
        self.water_stream = nn.Sequential(
            nn.Conv3d(1, width//2, kernel_size=3, padding=1),
            nn.GELU()
        )
        
        # Branch B: Bed Topography (Higher Magnitude)
        self.bed_stream = nn.Sequential(
            nn.Conv3d(1, width//2, kernel_size=3, padding=1),
            nn.GELU()
        )
        
        # Fusion Layer: Combines the two streams
        self.fusion = nn.Conv3d(width, width, kernel_size=1)
        
        # Deep Residual Body (Same as before)
        self.res_blocks = nn.ModuleList([
            ResidualDenseBlock3d(nf=width, gc=width//2) 
            for _ in range(num_blocks)
        ])
        
        self.projection = nn.Sequential(
            nn.Conv3d(width, width, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv3d(width, 1, kernel_size=1)
        )

    def forward(self, x):
        # x: [nb, 2, Pf, Pf, nt]
        water = x[:, 0:1, :, :, :] # Channel 0
        bed = x[:, 1:2, :, :, :]   # Channel 1
        
        # Process streams separately
        w_feat = self.water_stream(water)
        b_feat = self.bed_stream(bed)
        
        # Fuse and refine
        combined = torch.cat([w_feat, b_feat], dim=1)
        x = self.fusion(combined)
        
        for block in self.res_blocks:
            x = block(x)
            
        return self.projection(x)

In [4]:
# ------------------------------------------------------------------
# Usage and Testing for Multi-Stream Magnifier
# ------------------------------------------------------------------

if __name__ == "__main__":
    # Use GPU for testing if available (PSU Roar V100/A40)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Testing Multi-Stream Magnifier on device: {device}\n")
    print("=" * 70)

    # 1. Configuration matching your Hurricane Matthew setup
    nb = 32               # Batch size
    P_fine = 40           # N * f (e.g., 10 * 4)
    nt = 88               # Time steps
    C_in = 2              # Channel 0: Water, Channel 1: Bed Topography

    # 2. Initialize the Multi-Stream Model
    # width=48 allows 24 channels per stream before fusion
    model_ms = MultiStreamMagnifier(
        width=48, 
        num_blocks=4
    ).to(device)

    # 3. Create dummy input with different magnitudes to test stability
    # Channel 0 (Water): Small magnitude ~ [0, 5]
    water_dummy = torch.rand(nb, 1, P_fine, P_fine, nt, device=device) * 5.0
    # Channel 1 (Bed): Large magnitude ~ [0, 500]
    bed_dummy = torch.rand(nb, 1, P_fine, P_fine, nt, device=device) * 500.0
    
    patch_input = torch.cat([water_dummy, bed_dummy], dim=1)

    # 4. Forward Pass
    # The model internally splits the 2-channel input into separate streams
    output = model_ms(patch_input)
    
    # 5. Result Summary
    print(f"Input shape:  {patch_input.shape}")  # [32, 2, 40, 40, 88]
    print(f"Output shape: {output.shape}")     # [32, 1, 40, 40, 88]
    
    total_params = sum(p.numel() for p in model_ms.parameters())
    print(f"Total parameters: {total_params:,}")

    # Check that magnitudes didn't cause NaN outputs
    if torch.isnan(output).any():
        print("Warning: NaNs detected. Normalization may still be required.")
    else:
        print("Success: Model handled different magnitudes via separate streams.")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Testing Multi-Stream Magnifier on device: cuda

Input shape:  torch.Size([32, 2, 40, 40, 88])
Output shape: torch.Size([32, 1, 40, 40, 88])
Total parameters: 875,089
Success: Model handled different magnitudes via separate streams.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LightMagnifier(nn.Module):
    def __init__(self, width=32):
        super().__init__()
        
        # Branch A: Water Depth (Extracts flow features)
        self.water_stream = nn.Sequential(
            nn.Conv3d(1, width//2, kernel_size=3, padding=1),
            nn.GELU()
        )
        
        # Branch B: Bed Topography (Extracts terrain constraints)
        self.bed_stream = nn.Sequential(
            nn.Conv3d(1, width//2, kernel_size=3, padding=1),
            nn.GELU()
        )
        
        # Lightweight Bottleneck (No complex Dense Blocks)
        # Just 3 layers to learn the interaction between terrain and water
        self.refinement = nn.Sequential(
            nn.Conv3d(width, width, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv3d(width, width, kernel_size=3, padding=1),
            nn.GELU()
        )
        
        # Projection back to a single water depth channel
        self.projection = nn.Conv3d(width, 1, kernel_size=1)

    def forward(self, x):
        # x shape: [nb, 2, Pf, Pf, nt]
        # Store original interpolated water for Global Skip Connection
        base_water = x[:, 0:1, :, :, :] 
        bed = x[:, 1:2, :, :, :]
        
        # 1. Separate feature extraction
        w_feat = self.water_stream(base_water)
        b_feat = self.bed_stream(bed)
        
        # 2. Fusion
        combined = torch.cat([w_feat, b_feat], dim=1)
        
        # 3. Light Refinement
        x = self.refinement(combined)
        
        # 4. Global Skip Connection
        # The model predicts the DELTA (difference), and we add it to the base
        residual = self.projection(x)
        
        return base_water + residual

In [2]:
if __name__ == "__main__":
    # Use GPU for testing if available (PSU Roar cluster)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Testing LightMagnifier on device: {device}\n")
    print("=" * 70)

    # 1. Configuration matching your PSU dataset
    nb = 32               # Batch size
    P_fine = 50           # N=5 * f=10
    nt = 88               # Time steps
    
    # 2. Initialize the Light Model
    # width=32 is lighter and faster for 3D convolutions
    model_light = LightMagnifier(width=32).to(device)

    # 3. Create dummy input with different magnitudes
    # Water (Channel 0): ~0.0 to 5.0 meters
    water_dummy = torch.rand(nb, 1, P_fine, P_fine, nt, device=device) * 5.0
    # Bed (Channel 1): ~0.0 to 500.0 meters
    bed_dummy = torch.rand(nb, 1, P_fine, P_fine, nt, device=device) * 500.0
    
    patch_input = torch.cat([water_dummy, bed_dummy], dim=1)

    # 4. Forward Pass
    output = model_light(patch_input)
    
    # 5. Summary
    print(f"Input shape:  {patch_input.shape}")  
    print(f"Output shape: {output.shape}")     
    
    total_params = sum(p.numel() for p in model_light.parameters())
    print(f"Total parameters: {total_params:,}")

    # 6. Sanity Check for Global Skip Connection
    # The output should have a similar magnitude to the input water
    print(f"Mean Input Water:  {water_dummy.mean().item():.4f}")
    print(f"Mean Output Water: {output.mean().item():.4f}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Testing LightMagnifier on device: cuda

Input shape:  torch.Size([32, 2, 50, 50, 88])
Output shape: torch.Size([32, 1, 50, 50, 88])
Total parameters: 56,289
Mean Input Water:  2.4998
Mean Output Water: 1.3435
